In [1]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_core.output_parsers import StrOutputParser
import os
from langchain_neo4j import Neo4jGraph
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_experimental.graph_transformers import LLMGraphTransformer
from neo4j import GraphDatabase
from yfiles_jupyter_graphs import GraphWidget
from langchain_community.vectorstores import Neo4jVector
from langchain_community.document_loaders import TextLoader
from langchain_neo4j.vectorstores.neo4j_vector import remove_lucene_chars

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
graph = Neo4jGraph()

In [3]:
docs=TextLoader("dummytext.txt").load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=24)
documents = text_splitter.split_documents(documents=docs)

In [4]:
len(documents)

73

In [5]:
llm = ChatOllama(model='gemma3:4b', temperature=0)
llm_transformer = LLMGraphTransformer(llm=llm)

graph_documents = llm_transformer.convert_to_graph_documents(documents)

In [6]:
graph_documents[0]

GraphDocument(nodes=[Node(id="Amico'S Family", type='Family', properties={}), Node(id='Love', type='Concept', properties={}), Node(id='Tradition', type='Concept', properties={})], relationships=[Relationship(source=Node(id="Amico'S Family", type='Family', properties={}), target=Node(id='Love', type='Concept', properties={}), type='HAS', properties={})], source=Document(metadata={'source': 'dummytext.txt'}, page_content='1. The Story of Amicoâ€™s Family: A Legacy of Love and Tradition'))

In [8]:
graph.add_graph_documents(
    graph_documents, 
    baseEntityLabel=True,
    include_source=True
)

In [9]:
def showGraph():
    driver = GraphDatabase.driver(
        uri=os.environ['NEO4J_URI'],
        auth=(os.environ['NEO4J_USERNAME'], os.environ['NEO4J_PASSWORD']))
    session=driver.session()
    widget = GraphWidget(graph=session.run("MATCH (s)-[r:!MENTIONS]->(t) RETURN s, r, t").graph())
    widget.node_label_mapping = 'id'
    return widget

showGraph()

GraphWidget(layout=Layout(height='800px', width='100%'))

In [10]:
vector_index = Neo4jVector.from_existing_graph(
    OllamaEmbeddings(model='mxbai-embed-large:latest'),
    search_type='hybrid',
    node_label='Document',
    text_node_properties=['text'],
    embedding_node_property="embedding"
)

vector_retriever = vector_index.as_retriever()

In [11]:
class Entities(BaseModel):
    """Identifying information about entities."""
    
    names: list[str] = Field(
        ..., 
        description="All the person, organization, or bussiness entities that appear in the text"
    )

prompt = ChatPromptTemplate.from_messages(
    [("system", "You are extractingn organization and person entities from the text."),
     ("human", "Use the given format to extract information from the following"
      "input: {question}")]
)

entity_chain = prompt | llm.with_structured_output(Entities)

In [12]:
entity_chain.invoke({"question":"Who are Nonna Lucia and Giovanni Caruso?"}).names


['Nonna Lucia', 'Giovanni Caruso']

In [13]:
def generate_full_text_query(input: str) -> str:
    words = [el for el in remove_lucene_chars(input).split() if el]
    if not words:
        return ""
    full_text_query = " AND ".join([f"{word}~2" for word in words])
    print(f"Generated Query: {full_text_query}")
    return full_text_query.strip()


def graph_retriever(question: str) -> str:
    """
    Collects the neighborhood of entities mentioned in the question
    """
    
    result = ""
    entities = entity_chain.invoke({"question": question})

    if not entities or not getattr(entities, "names", None):
        return ""

    for entity in entities.names:
        response = graph.query(
            """
            CALL db.index.fulltext.queryNodes('entity', $query, {limit: 2})
            YIELD node, score
            CALL {
                WITH node
                MATCH (node)-[r]->(neighbor)
                WHERE type(r) <> 'MENTIONS'
                RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output

                UNION ALL

                WITH node
                MATCH (node)<-[r]-(neighbor)
                WHERE type(r) <> 'MENTIONS'
                RETURN neighbor.id + ' - ' + type(r) + ' -> ' + node.id AS output
            }
            RETURN output LIMIT 50
            """,
            {"query": generate_full_text_query(entity)},
        )

        result += "\n".join(
            [el.get("output", "") for el in response if el.get("output")]
        ) + "\n"

    return result.strip()

In [14]:
print(graph_retriever("Who is Nonna Lucia?"))

Generated Query: Nonna~2 AND Lucia~2


ClientError: {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `db.index.fulltext.queryNodes`: Caused by: java.lang.IllegalArgumentException: There is no such fulltext schema index: entity} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Failed to invoke procedure `db.index.fulltext.queryNodes`: Caused by: java.lang.IllegalArgumentException: There is no such fulltext schema index: entity}